In [29]:
# =========================================================
# STAGE 0: IMPORTS AND BASIC SETUP
# =========================================================
import os
import re
import numpy as np
import torch
import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Optional: disable wandb logging
os.environ["WANDB_DISABLED"] = "true"

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [30]:
# ============================================================
# STAGE 1: LOAD DATASET
# ============================================================

import pandas as pd
from datasets import Dataset

# ------------------------------------------------------------
# STEP 1.1: LOAD CSV FROM KAGGLE INPUT PATH
# ------------------------------------------------------------

csv_path = "/kaggle/input/datasets/kezheonglim/new-21-per-serve/dataset_filtered_20_plus_target_per_serve.csv"
df = pd.read_csv(csv_path)

print("✅ CSV loaded successfully.")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3))


# ------------------------------------------------------------
# STEP 1.2: DEFINE LABEL COLUMN CLEARLY
# ------------------------------------------------------------

label_column = "sugar_per_serving_g"   # change here if needed

if label_column not in df.columns:
    raise ValueError(
        f"❌ Label column '{label_column}' not found.\n"
        f"Available columns: {df.columns.tolist()}"
    )

print(f"\n✅ Using label column: {label_column}")


# ------------------------------------------------------------
# STEP 1.3: DEFINE FEATURE COLUMNS
# Use all columns except the label column
# ------------------------------------------------------------

feature_columns = [col for col in df.columns if col != label_column]

print(f"✅ Number of feature columns: {len(feature_columns)}")
print("✅ Feature columns:", feature_columns)


# ------------------------------------------------------------
# STEP 1.4: OPTIONAL - FILL MISSING VALUES
# Text columns -> empty string
# Numeric columns -> median
# ------------------------------------------------------------

for col in feature_columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("").astype(str)
    else:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())

df[label_column] = pd.to_numeric(df[label_column], errors="coerce")
df = df.dropna(subset=[label_column]).copy()

print("\n✅ Missing values handled.")


# ------------------------------------------------------------
# STEP 1.5: COMBINE ALL FEATURES INTO ONE TEXT COLUMN
# This allows a text model to use all columns
# ------------------------------------------------------------

def combine_features(row):
    parts = []
    for col in feature_columns:
        parts.append(f"{col}: {row[col]}")
    return " | ".join(parts)

df["text"] = df.apply(combine_features, axis=1)
df["labels"] = df[label_column].astype(float)

print("\n✅ Combined all feature columns into 'text'")
print(df[["text", "labels"]].head(2))


# ------------------------------------------------------------
# STEP 1.6: KEEP ONLY MODEL-READY COLUMNS
# ------------------------------------------------------------

df_model = df[["text", "labels"]].copy()

print("\n✅ Final model dataframe ready.")
print("Final columns:", df_model.columns.tolist())
print(df_model.head(3))


# ------------------------------------------------------------
# STEP 1.7: CONVERT DATAFRAME TO HUGGING FACE DATASET
# ------------------------------------------------------------

dataset = Dataset.from_pandas(df_model)

if "__index_level_0__" in dataset.column_names:
    dataset = dataset.remove_columns(["__index_level_0__"])

print("\n✅ Converted to Hugging Face Dataset.")
print("Dataset columns:", dataset.column_names)


# ------------------------------------------------------------
# STEP 1.8: SPLIT INTO TRAIN / TEST
# ------------------------------------------------------------

dataset = dataset.train_test_split(test_size=0.2, seed=42)

print("\n✅ Train-test split completed.")
print("Train size:", len(dataset["train"]))
print("Test size :", len(dataset["test"]))


# ------------------------------------------------------------
# STEP 1.9: CREATE TRAIN / TEST REFERENCES
# ------------------------------------------------------------

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print("\n✅ Stage 1 completed successfully.")
print("Train samples:", len(train_dataset))
print("Test samples :", len(test_dataset))


# ------------------------------------------------------------
# STEP 1.10: SANITY CHECK
# ------------------------------------------------------------

print("\n✅ Sample records:")
for i in range(min(3, len(train_dataset))):
    print(f"\nSample {i+1}")
    print("Text  :", train_dataset[i]["text"][:500])  # print first 500 chars
    print("Label :", train_dataset[i]["labels"])

✅ CSV loaded successfully.
Shape: (27638, 21)
Columns: ['recipe_name', 'ingredients', 'servings', 'STARCH', 'FIBTG', 'WATER', 'FAT', 'PROCNT', 'CHOCDF', 'ENERC_KCAL', 'SUCS', 'GLUS', 'FRUS', 'LACS', 'MALS', 'GALS', 'CHOLE', 'ASH', 'CAFFN', 'NA', 'sugar_per_serving_g']
                           recipe_name  \
0                     Mushroom Risotto   
1            Filipino BBQ Pork Skewers   
2  Mushroom and Roasted Garlic Risotto   

                                         ingredients  servings  STARCH  FIBTG  \
0  2 cups Baby Bella mushrooms, sliced, 2 cups ar...       6.0    0.02   2.85   
1  2.5 lb pork country style ribs, all fat trimme...       4.0     NaN   3.25   
2  2 whole garlic heads, 2 tablespoons plus 2 tea...       1.0     NaN  34.70   

     WATER    FAT  PROCNT  CHOCDF  ENERC_KCAL  ...  GLUS  FRUS  LACS  MALS  \
0   400.14   8.54   19.14   75.34      476.93  ...  0.34  0.02   NaN   NaN   
1   308.85  34.93   56.97   26.21      650.03  ...  1.40  2.25   NaN   NaN   
2  

In [24]:
# ============================================================
# STAGE 2: EDA
# - columns were loaded correctly
# - text and numeric fields are in the right format
# - there are repeated recipes that may distort training
# ============================================================

# ==========================
# A. Basic structure check
# ==========================
# print(df.shape)
# print(df.columns.tolist())
# print(df.dtypes)
# print(df.duplicated().sum())
# print(df["recipe_name"].duplicated().sum())

# ==========================
# B. Missing value analysis
# ==========================
# missing = df.isnull().sum().sort_values(ascending=False)
# missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)

# missing_df = pd.DataFrame({
#     "missing_count": missing,
#     "missing_pct": missing_pct
# })
# print(missing_df)


# ==========================
# C. Invalid value check
# ==========================
num_cols = ["servings", "STARCH", "FIBTG", "WATER", "FAT", "PROCNT",
            "CHOCDF", "ENERC_KCAL", "SUCS", "GLUS", "FRUS",
            "LACS", "MALS", "GALS", "CHOLE", "ASH", "CAFFN", "NA",
            "sugar_per_serving_g"]

print(df[num_cols].describe().T)

                       count        mean          std   min       25%  \
servings             27626.0    7.589191    19.006718  1.00    4.0000   
STARCH               12433.0   11.745644    32.192451  0.00    0.1200   
FIBTG                24740.0    5.810842    16.438600  0.00    1.5700   
WATER                23838.0  235.060269   309.174890  0.01   90.3825   
FAT                  24773.0   25.159512    87.680107  0.00    7.7600   
PROCNT               24937.0   25.023187   205.907794  0.00    4.8100   
CHOCDF               24975.0   45.805885   121.627828  0.00   14.4300   
ENERC_KCAL           24985.0  501.929264  1590.709683  0.90  205.9700   
SUCS                 18785.0    5.973821    22.338334  0.00    0.1800   
GLUS                 19673.0    1.833848     4.128066  0.00    0.3200   
FRUS                 19086.0    1.807479     4.144824  0.00    0.2700   
LACS                  2718.0    1.748863     3.885727  0.01    0.3000   
MALS                  4055.0    0.775714     1.3648

In [31]:
# ============================================================
# STAGE 2: PREPROCESSING (TEXT NORMALIZATION + SUGAR STANDARDIZATION)
# ============================================================

import unicodedata
from fractions import Fraction
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# STEP 2.0: CONFIGURATION
# ------------------------------------------------------------

# choose one target only
TARGET_COLUMN = "labels"          # already created in Stage 1
USE_LOG_TARGET = False            # set True to use log1p(labels) instead of raw clipped labels
CLIP_FEATURES = True              # clip numeric features using train percentiles
NORMALIZE_FEATURES = True         # standardize numeric features
PRINT_EXTREME_ROWS = True         # inspect suspicious rows

# numeric columns that exist in your dataset and should be used
CANDIDATE_NUMERIC_FEATURES = [
    "servings", "STARCH", "FIBTG", "WATER", "FAT", "PROCNT",
    "CHOCDF", "ENERC_KCAL", "SUCS", "GLUS", "FRUS",
    "LACS", "MALS", "GALS", "CHOLE", "ASH", "CAFFN", "NA"
]

# text columns you want to preserve
TEXT_SOURCE_COLUMN = "text"       # from Stage 1, this is the model input text source

# ------------------------------------------------------------
# STEP 2.1: HANDLE SPECIAL CHARACTERS & FRACTIONS
# Convert strange characters like “½”, “å”, “¨” → clean ASCII
# ------------------------------------------------------------

UNICODE_FRACTIONS = {
    "¼": "1/4", "½": "1/2", "¾": "3/4",
    "⅐": "1/7", "⅑": "1/9", "⅒": "1/10",
    "⅓": "1/3", "⅔": "2/3",
    "⅕": "1/5", "⅖": "2/5", "⅗": "3/5", "⅘": "4/5",
    "⅙": "1/6", "⅚": "5/6",
    "⅛": "1/8", "⅜": "3/8", "⅝": "5/8", "⅞": "7/8",
}

def strip_accents_and_symbols(text: str) -> str:
    """Remove weird unicode characters and normalize text."""
    if text is None:
        return ""

    text = str(text)

    # Replace unicode fractions
    for bad, good in UNICODE_FRACTIONS.items():
        text = text.replace(bad, good)

    # Normalize unicode → ASCII
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")

    # Clean leftover symbols
    text = text.replace("`", "").replace("´", "").replace("¨", "")
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = text.replace("–", "-").replace("—", "-")

    return text


# ------------------------------------------------------------
# STEP 2.2: NORMALIZE BASIC TEXT
# Standardize units & clean format (DO NOT REMOVE NUMBERS)
# ------------------------------------------------------------

GENERAL_UNIT_NORMALIZATION = {
    "tablespoons": "tbsp", "tablespoon": "tbsp", "tbs": "tbsp", "tbl": "tbsp",
    "teaspoons": "tsp", "teaspoon": "tsp",
    "ounces": "oz", "ounce": "oz",
    "pounds": "lb", "pound": "lb", "lbs": "lb",
    "grams": "g", "gram": "g",
    "kilograms": "kg", "kilogram": "kg",
    "cups": "cup",
}

def normalize_basic_text(text: str) -> str:
    """Lowercase, normalize units, remove noise."""
    text = strip_accents_and_symbols(text)
    text = text.lower()

    # Remove URLs if any
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Normalize units (tablespoons → tbsp)
    for src, tgt in GENERAL_UNIT_NORMALIZATION.items():
        text = re.sub(rf"\b{re.escape(src)}\b", tgt, text)

    # Keep numbers, fractions, units → remove only unwanted symbols
    text = re.sub(r"[^a-z0-9\s,./()\-]", " ", text)

    # Clean spacing
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ------------------------------------------------------------
# STEP 2.3: PARSE NUMERIC QUANTITIES
# Support formats:
# - 1
# - 1.5
# - 1/2
# - 1 1/2
# ------------------------------------------------------------

def parse_quantity(qty_text: str) -> float:
    qty_text = qty_text.strip()

    # Mixed fraction: 1 1/2
    if re.fullmatch(r"\d+\s+\d+/\d+", qty_text):
        whole, frac = qty_text.split()
        return float(whole) + float(Fraction(frac))

    # Fraction: 1/2
    if re.fullmatch(r"\d+/\d+", qty_text):
        return float(Fraction(qty_text))

    # Decimal / integer
    return float(qty_text)


# ------------------------------------------------------------
# STEP 2.4: STANDARDIZE SUGAR QUANTITY INTO GRAMS
# Keep ingredient identity + normalized quantity
# ------------------------------------------------------------

SUGAR_WORDS = [
    "brown sugar", "white sugar", "caster sugar", "granulated sugar",
    "icing sugar", "powdered sugar", "confectioners sugar",
    "raw sugar", "demerara sugar", "coconut sugar", "palm sugar",
    "sugar", 
]

# high sweeteners
HIGH_SWEETENER_WORDS = [
    "honey", "maple syrup", "corn syrup", "golden syrup", "chocolate syrup",
    "syrup", "molasses", "condensed milk", "sweetened condensed milk", "caramel",
]

# medium sweeteners
MID_SWEETENER_WORDS = [
    "banana", "raisins", "raisin", "dates", "date",
    "apple", "mango", "pineapple", "grape", "grapes",
    "pear", "pears",
]

# ------------------------------------------------------------
# STEP 2.5: UNIT CONVERSION TABLES
# ------------------------------------------------------------

# fallback (VERY IMPORTANT)
DEFAULT_UNIT_TO_G = {
    "g": 1, "kg": 1000, "oz": 28.35, "lb": 453.59,
    "tsp": 5, "tbsp": 15, "cup": 240
}

SUGAR_UNIT_TO_G = {
    "tsp": 4.2, "tbsp": 12.5, "cup": 200
}

HIGH_SWEETENER_UNIT_TO_G = {
    "tsp": 7, "tbsp": 21, "cup": 340
}

MID_SWEETENER_UNIT_TO_G = {
    "banana": {"cup": 150, "tbsp": 15, "tsp": 5},
    "raisins": {"cup": 165, "tbsp": 10, "tsp": 3},
    "apple": {"cup": 125, "tbsp": 10, "tsp": 3},
    "mango": {"cup": 165, "tbsp": 12, "tsp": 4},
    "pineapple": {"cup": 165, "tbsp": 12, "tsp": 4},
    "dates": {"cup": 160, "tbsp": 12, "tsp": 4},
}

# ------------------------------------------------------------
# STEP 2.6: CORE CONVERSION FUNCTION (FIXED)
# ------------------------------------------------------------

def convert_to_grams(qty, unit, ingredient):

    # sugar
    if ingredient in SUGAR_WORDS:
        return qty * SUGAR_UNIT_TO_G.get(unit, DEFAULT_UNIT_TO_G.get(unit, 0))

    # high sweet
    if ingredient in HIGH_SWEETENER_WORDS:
        return qty * HIGH_SWEETENER_UNIT_TO_G.get(unit, DEFAULT_UNIT_TO_G.get(unit, 0))

    # mid sweet (ingredient-specific)
    if ingredient in MID_SWEETENER_UNIT_TO_G:
        table = MID_SWEETENER_UNIT_TO_G[ingredient]
        return qty * table.get(unit, DEFAULT_UNIT_TO_G.get(unit, 0))

    # fallback
    return qty * DEFAULT_UNIT_TO_G.get(unit, 0)
    

# ------------------------------------------------------------
# STEP 2.7: STANDARDIZE TEXT (MAIN LOGIC)
# ------------------------------------------------------------

UNIT_PATTERN = r"(g|kg|oz|lb|tsp|tbsp|cup)"

def standardize_sugars_and_sweeteners(text: str) -> str:

    all_words = SUGAR_WORDS + HIGH_SWEETENER_WORDS + MID_SWEETENER_WORDS
    all_words = sorted(all_words, key=len, reverse=True)
    word_pattern = "|".join(map(re.escape, all_words))

    pattern = re.compile(
        rf"(?P<qty>\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)\s*"
        rf"(?P<unit>{UNIT_PATTERN})\s+"
        rf"(?P<name>{word_pattern})\b"
    )

    def repl(match):
        qty = parse_quantity(match.group("qty"))
        unit = match.group("unit")
        name = match.group("name")

        grams = convert_to_grams(qty, unit, name)

        return f"{name.replace(' ', '_')}_qty_g_{int(round(grams))} {name}"

    return pattern.sub(repl, text)

# ------------------------------------------------------------
# STEP 2.8: FINAL CLEANING PIPELINE
# ------------------------------------------------------------

def clean_ingredient_text(text: str) -> str:
    if text is None:
        return ""

    text = normalize_basic_text(text)
    text = standardize_sugars_and_sweeteners(text)

    text = re.sub(r"[()]", " ", text)
    text = re.sub(r"\s*,\s*", ", ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

# ------------------------------------------------------------
# STEP 2.9: DETECT NUMERIC FEATURE COLUMNS
# ------------------------------------------------------------

train_columns = dataset["train"].column_names
numeric_feature_cols = [col for col in CANDIDATE_NUMERIC_FEATURES if col in train_columns]

print("✅ Numeric feature columns found:", numeric_feature_cols)


# ------------------------------------------------------------
# STEP 2.10: APPLY TEXT PREPROCESSING + NUMERIC CASTING
# ------------------------------------------------------------

def preprocess_example(example):
    # clean text
    example["clean_text"] = clean_ingredient_text(example.get(TEXT_SOURCE_COLUMN, ""))

    # numeric columns
    for col in numeric_feature_cols:
        value = example.get(col, None)
        try:
            example[col] = float(value) if value is not None and value != "" else np.nan
        except:
            example[col] = np.nan

    # label
    try:
        example[TARGET_COLUMN] = float(example[TARGET_COLUMN])
    except:
        example[TARGET_COLUMN] = np.nan

    return example

dataset = dataset.map(preprocess_example)


# ------------------------------------------------------------
# STEP 2.11: FILTER INVALID ROWS
# ------------------------------------------------------------

dataset = dataset.filter(
    lambda x: x["clean_text"] is not None and len(x["clean_text"]) > 3 and x[TARGET_COLUMN] is not None
)

print("✅ Invalid rows filtered.")


# ------------------------------------------------------------
# STEP 2.12: INSPECT EXTREME ROWS
# ------------------------------------------------------------

train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

if PRINT_EXTREME_ROWS:
    inspect_cols = [col for col in ["recipe_name", "ingredients", TEXT_SOURCE_COLUMN, "clean_text",
                                    "servings", "ENERC_KCAL", "PROCNT", "FAT", TARGET_COLUMN]
                    if col in train_df.columns]

    print("\n================ EXTREME ROW INSPECTION (TRAIN) ================\n")

    if "servings" in train_df.columns:
        print("Top 10 highest servings:")
        print(train_df.sort_values("servings", ascending=False)[inspect_cols].head(10))

    if "ENERC_KCAL" in train_df.columns:
        print("\nTop 10 highest ENERC_KCAL:")
        print(train_df.sort_values("ENERC_KCAL", ascending=False)[inspect_cols].head(10))

    print(f"\nTop 10 highest {TARGET_COLUMN}:")
    print(train_df.sort_values(TARGET_COLUMN, ascending=False)[inspect_cols].head(10))


# ------------------------------------------------------------
# STEP 2.13: FILL MISSING NUMERIC FEATURES USING TRAIN MEDIAN
# ------------------------------------------------------------

feature_medians = {}
for col in numeric_feature_cols:
    median_value = train_df[col].median()
    feature_medians[col] = median_value
    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)

print("✅ Missing numeric features filled with train medians.")


# ------------------------------------------------------------
# STEP 2.14: CLIP NUMERIC FEATURE OUTLIERS
# Using train 1st and 99th percentile
# ------------------------------------------------------------

feature_clip_bounds = {}

if CLIP_FEATURES:
    for col in numeric_feature_cols:
        lower = train_df[col].quantile(0.01)
        upper = train_df[col].quantile(0.99)
        feature_clip_bounds[col] = (lower, upper)

        train_df[col] = train_df[col].clip(lower, upper)
        test_df[col] = test_df[col].clip(lower, upper)

    print("✅ Numeric features clipped using train 1st/99th percentile.")


# ------------------------------------------------------------
# STEP 2.15: TARGET TREATMENT
# Option A: clip raw labels
# Option B: log-transform labels
# ------------------------------------------------------------

train_labels = train_df[TARGET_COLUMN].values

if USE_LOG_TARGET:
    train_df["labels_for_model"] = np.log1p(train_df[TARGET_COLUMN].clip(lower=0))
    test_df["labels_for_model"] = np.log1p(test_df[TARGET_COLUMN].clip(lower=0))
    print("✅ Using log1p(labels) as training target: labels_for_model")
else:
    lower_bound = np.percentile(train_labels, 1)
    upper_bound = np.percentile(train_labels, 99)

    train_df["labels_for_model"] = np.clip(train_df[TARGET_COLUMN], lower_bound, upper_bound)
    test_df["labels_for_model"] = np.clip(test_df[TARGET_COLUMN], lower_bound, upper_bound)

    print(f"✅ Using clipped raw labels as training target: labels_for_model")
    print(f"   lower_bound={lower_bound:.4f}, upper_bound={upper_bound:.4f}")


# ------------------------------------------------------------
# STEP 2.16: NORMALIZE NUMERIC FEATURES
# ------------------------------------------------------------

if NORMALIZE_FEATURES and len(numeric_feature_cols) > 0:
    scaler = StandardScaler()
    train_df[numeric_feature_cols] = scaler.fit_transform(train_df[numeric_feature_cols])
    test_df[numeric_feature_cols] = scaler.transform(test_df[numeric_feature_cols])
    print("✅ Numeric features normalized with StandardScaler.")


# ------------------------------------------------------------
# STEP 2.17: REBUILD HUGGING FACE DATASET
# Keep original text, clean_text, numeric features, and model label
# ------------------------------------------------------------

# IMPORTANT:
# - keep "clean_text" for tokenizer input later
# - keep "labels_for_model" as final target for training
# - keep numeric features if you want hybrid input later

from datasets import Dataset, DatasetDict

train_dataset_raw = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset_raw = Dataset.from_pandas(test_df.reset_index(drop=True))

# remove auto index columns if created
for split_name, split_ds in [("train", train_dataset_raw), ("test", test_dataset_raw)]:
    if "__index_level_0__" in split_ds.column_names:
        if split_name == "train":
            train_dataset_raw = split_ds.remove_columns(["__index_level_0__"])
        else:
            test_dataset_raw = split_ds.remove_columns(["__index_level_0__"])

dataset = DatasetDict({
    "train": train_dataset_raw,
    "test": test_dataset_raw
})


# ------------------------------------------------------------
# STEP 2.18: SHUFFLE FINAL DATASET
# ------------------------------------------------------------

train_dataset_raw = dataset["train"].shuffle(seed=42)
test_dataset_raw = dataset["test"].shuffle(seed=42)

print("\n✅ Final dataset ready for training")
print("Train size:", len(train_dataset_raw))
print("Test size :", len(test_dataset_raw))


# ------------------------------------------------------------
# STEP 2.19: SANITY CHECK
# ------------------------------------------------------------

print("\n✅ Sample after preprocessing:")
for i in range(min(5, len(train_dataset_raw))):
    print(f"\nSample {i+1}")
    if "text" in train_dataset_raw.column_names:
        print("RAW TEXT      :", train_dataset_raw[i]["text"][:200])
    print("CLEAN TEXT    :", train_dataset_raw[i]["clean_text"][:200])
    print("MODEL LABEL   :", train_dataset_raw[i]["labels_for_model"])

    for col in numeric_feature_cols[:5]:
        print(f"{col:14}:", train_dataset_raw[i][col])


✅ Numeric feature columns found: []


Map:   0%|          | 0/19904 [00:00<?, ? examples/s]

Map:   0%|          | 0/4976 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19904 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4976 [00:00<?, ? examples/s]

✅ Invalid rows filtered.

================ EXTREME ROW INSPECTION (TRAIN) ================


Top 10 highest labels:
                                                    text  \
5947   recipe_name: Double-Chocolate Cupcakes | ingre...   
4554   recipe_name: Triple Layer Chocolate Chip-Fudge...   
12551  recipe_name: Peanut Butter Chocolate Bars | in...   
10725  recipe_name: Peppermint Brownies | ingredients...   
775    recipe_name: Sugar Cookies | ingredients: 2/3 ...   
4977   recipe_name: Coconut Cream Pie | ingredients: ...   
6409   recipe_name: Gramercy Tavern's Monkey Bread | ...   
6741   recipe_name: Davy Crockett Cookies | ingredien...   
10875  recipe_name: Uncle Earl's NC BBQ Sauce | ingre...   
7842   recipe_name: Flourless Chewy Cinnamon Sugar Pe...   

                                              clean_text   labels  
5947   recipe name double-chocolate cupcakes ingredie...  4161.46  
4554   recipe name triple layer chocolate chip-fudge ...   734.69  
12551  recipe name 

In [32]:
# =========================================================
# STAGE 3: HELPER FUNCTIONS
# =========================================================
# Regression metrics because your current setup predicts a continuous sugar score.
def compute_metrics(eval_pred):
    """
    Compute evaluation metrics for regression task.

    IMPORTANT:
    - This version assumes NO log-transform was applied to labels.
    - DO NOT use np.expm1() here unless you used np.log1p() during training.
    """

    # Unpack predictions and true labels
    predictions, labels = eval_pred

    # Remove extra dimensions (e.g., shape [batch, 1] → [batch])
    preds = np.squeeze(predictions)
    labels = np.squeeze(labels)

    # -----------------------------
    # Debug check (optional)
    # -----------------------------
    # Helps detect exploding values early
    # Uncomment if needed:
    # print("Pred min/max:", np.min(preds), np.max(preds))
    # print("Label min/max:", np.min(labels), np.max(labels))

    # -----------------------------
    # Metric calculations
    # -----------------------------

    # MAE: average absolute error
    mae = mean_absolute_error(labels, preds)

    # MSE: average squared error
    mse = mean_squared_error(labels, preds)

    # RMSE: square root of MSE (penalizes large errors more)
    rmse = np.sqrt(mse)

    # R²: how well model explains variance
    r2 = r2_score(labels, preds)

    # -----------------------------
    # Return results
    # -----------------------------
    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
    }


def tokenize_dataset(dataset_dict, tokenizer, max_length=192):
    """
    Tokenize dataset using the provided tokenizer.
    """
    def tokenize_function(examples):
        return tokenizer(
            examples["clean_text"],
            padding="max_length",
            truncation=True,
            max_length=max_length,
        )

    tokenized_train = dataset_dict["train"].map(tokenize_function, batched=True)
    tokenized_test = dataset_dict["test"].map(tokenize_function, batched=True)

    tokenized_train = tokenized_train.with_format("torch")
    tokenized_test = tokenized_test.with_format("torch")

    return tokenized_train, tokenized_test


def build_trainer(model_name, output_dir, train_dataset, test_dataset):
    """
    Build tokenizer, model, training args, and trainer for one model.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Re-tokenize using this model's tokenizer
    dataset_dict = {
        "train": train_dataset,
        "test": test_dataset,
    }
    tokenized_train, tokenized_test = tokenize_dataset(dataset_dict, tokenizer)

    # Load model for regression
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1
    )
    model.config.problem_type = "regression"
    model.to(device)

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        warmup_steps=100,
        max_steps=1500,              # change if needed, default 2000
        learning_rate=2e-5,
        fp16=torch.cuda.is_available(),
        logging_steps=20,
        eval_strategy="steps",
        save_strategy="steps",
        eval_steps=100,
        save_steps=100,
        save_total_limit=3,
        load_best_model_at_end=True,
        push_to_hub=False,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        # tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    return trainer, tokenizer, tokenized_test

In [33]:
# =========================================================
# STAGE 4: TRAIN DISTILBERT
# =========================================================
distilbert_name = "distilbert-base-uncased"
distilbert_output = "distilbert_sugar_model"

distilbert_trainer, distilbert_tokenizer, distilbert_test_dataset = build_trainer(
    model_name=distilbert_name,
    output_dir=distilbert_output,
    train_dataset=train_dataset_raw,
    test_dataset=test_dataset_raw,
)

# Train DistilBERT
distilbert_trainer.train()

# Save final DistilBERT model
distilbert_trainer.save_model(distilbert_output)
distilbert_tokenizer.save_pretrained(distilbert_output)

# Evaluate DistilBERT
distilbert_results = distilbert_trainer.evaluate()
print("DistilBERT Results:", distilbert_results)

Map:   0%|          | 0/19904 [00:00<?, ? examples/s]

Map:   0%|          | 0/4976 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, 

Step,Training Loss,Validation Loss,Mae,Rmse,R2
100,13810.751562,9646.458008,5.451166,69.449471,-0.001211
200,219008.900000,9621.020508,4.863459,69.357846,0.001429
300,6580.088281,9581.036133,4.586017,69.213569,0.005579
400,5958.441797,9565.145508,4.584032,69.156148,0.007228
500,6355.733984,9535.453125,4.241145,69.048730,0.010310
600,5203.415234,9511.526367,4.191018,68.962038,0.012794
700,6796.907812,9497.797852,4.186133,68.912255,0.014218
800,15262.546875,9458.264648,3.964760,68.768687,0.018322
900,7922.046875,9440.250000,3.885120,68.703169,0.020191
1000,8660.017187,9418.529297,3.902601,68.624086,0.022446


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


KeyboardInterrupt: 

In [ ]:
# =========================================================
# STAGE 5: TRAIN BERT
# =========================================================
bert_name = "bert-base-uncased"
bert_output = "bert_sugar_model"

bert_trainer, bert_tokenizer, bert_test_dataset = build_trainer(
    model_name=bert_name,
    output_dir=bert_output,
    train_dataset=train_dataset_raw,
    test_dataset=test_dataset_raw,
)

# Train BERT
bert_trainer.train()

# Save final BERT model
bert_trainer.save_model(bert_output)
bert_tokenizer.save_pretrained(bert_output)

# Evaluate BERT
bert_results = bert_trainer.evaluate()
print("BERT Results:", bert_results)

In [ ]:
# =========================================================
# STAGE 6: COMPARE BOTH MODELS
# =========================================================
print("\n========== MODEL COMPARISON ==========")
print("DistilBERT MAE :", distilbert_results.get("eval_mae"))
print("DistilBERT RMSE:", distilbert_results.get("eval_rmse"))
print("DistilBERT R2:", distilbert_results.get("eval_r2"))

print("BERT MAE       :", bert_results.get("eval_mae"))
print("BERT RMSE      :", bert_results.get("eval_rmse"))
print("BERT R2        :", bert_results.get("eval_r2"))

In [ ]:
# =========================================================
# STAGE 7: TEST BOTH MODELS ON ONE SAMPLE
# =========================================================
sample_text = "1 cup sugar, 2 tablespoons honey, 1 cup milk, 1 teaspoon vanilla extract"

def predict_single_text(model, tokenizer, text):
    """
    Run inference for one text input.
    """
    model.eval()

    encoded = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=192,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        prediction = outputs.logits.squeeze().item()

    return prediction

sample_text_clean = clean_ingredient_text(sample_text)

distilbert_pred = predict_single_text(
    distilbert_trainer.model,
    distilbert_tokenizer,
    sample_text_clean
)

bert_pred = predict_single_text(
    bert_trainer.model,
    bert_tokenizer,
    sample_text_clean
)

print("\n========== SAMPLE PREDICTION ==========")
print("Input raw text:", sample_text)
print("Input cleaned text:", sample_text_clean)
print("DistilBERT predicted sugar score:", distilbert_pred)
print("BERT predicted sugar score      :", bert_pred)